# Detecting Collapse Before It Ruins the Run

**Topic:** Representational collapse &nbsp;·&nbsp; **Difficulty:** Medium

Non-contrastive methods have no negatives, so nothing stops the encoder from
mapping every input to the same vector: a perfect loss and a useless
representation. The loss curve looks *great* while this happens, which is why you
need a separate diagnostic.

Effective rank is the standard one: how many directions the embeddings actually
use. Compute it for four checkpoints, three of which are collapsing in different
ways.

## The data (created in the next cell)

- `embeddings`, a list of four `(500, 64)` matrices from four training runs:
  healthy, fully collapsed, dimensionally collapsed (rank 3), and one where all
  vectors point the same way but have varied norms.

## Your task

With **NumPy only**, for each embedding matrix:

1. Take singular values **without centring** and normalize them to a probability
   distribution `p_i = s_i / sum(s)` (drop singular values that are exactly zero
   before the entropy, since 0 log 0 is NaN, for example `s = s[s > 1e-12]`).
   Centring would subtract the mean embedding
   and hide exactly the failure you are looking for: a fully collapsed run
   looks full-rank once its constant is removed.
2. `effective_rank = exp(entropy(p))` in nats, the standard definition.
3. Also compute `std_of_normalized`, the mean per-dimension standard deviation
   after L2-normalizing each row.
4. Set `answers` to a dict with `"eff_rank"` (length-4), `"std_norm"` (length-4),
   and `"n_collapsed"`, the count of runs with effective rank below 5.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the healthy run has much higher effective rank than the others and that
the std diagnostic is more than 10 times smaller for the fully collapsed run
than for the healthy run.

In [ ]:
import numpy as np

rng = np.random.default_rng(3106)
n, d = 500, 64

healthy = rng.normal(0, 1, (n, d))

collapsed = np.tile(rng.normal(0, 1, (1, d)), (n, 1)) + rng.normal(0, 1e-4, (n, d))

basis = rng.normal(0, 1, (3, d))
dim_collapsed = rng.normal(0, 1, (n, 3)) @ basis

direction = rng.normal(0, 1, d)
direction /= np.linalg.norm(direction)
varied_norm = rng.uniform(0.2, 5.0, (n, 1)) * direction[None, :]

embeddings = [healthy, collapsed, dim_collapsed, varied_norm]
print(f"four checkpoints of {n}x{d}: healthy, collapsed, rank-3, and same-direction")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("eff_rank", [4], 67.89666898658184, -13.284916531700427, 1e-06),
    ("std_norm", [4], 0.2380149945475633, -0.19255666080439093, 1e-06),
    ("n_collapsed", [], 3.0, -0.5023140444434975, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "eff_rank" in answers:
    _e = np.asarray(answers["eff_rank"], dtype=float)
    _s = np.asarray(answers["std_norm"], dtype=float)
    _healthy = _e[0] > 20 and _e[1] < 5 and _e[2] < 5 and _e[3] < 5
    _agree = _s[0] > 10 * _s[1]
    print(f"  {'healthy rank is high':<24} {'ok' if _healthy else 'wrong'}  "
          f"(ranks {np.round(_e, 1).tolist()})")
    print(f"  {'diagnostics agree':<24} {'ok' if _agree else 'wrong'}  "
          f"(std {np.round(_s, 4).tolist()})")
    _ok = _ok and _healthy and _agree

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Centring is the trap: subtract the mean and the fully-collapsed matrix becomes
its own residual noise, which is full rank and scores 63. Leaving the mean in
keeps the collapse visible as a single dominant singular value. Beyond that,
effective rank uses the entropy of the singular-value spectrum, so it is a
*soft* count: a matrix using 60 directions equally scores near 60, and one whose
energy sits in 3 directions scores near 3, regardless of the numerical rank. The
same-direction case is the interesting one: its rows differ (different norms), so
naive variance checks look fine, but every row is a multiple of one vector, so
after normalizing they are identical and the effective rank is 1. That is exactly
the failure mode a norm-blind diagnostic misses.

```python
import numpy as np

def eff_rank(Z):
    s = np.linalg.svd(Z, compute_uv=False)
    s = s[s > 1e-12]
    if s.size == 0:
        return 1.0
    p = s / s.sum()
    H = -np.sum(p * np.log(p))
    return float(np.exp(H))

def std_norm(Z):
    U = Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-12)
    return float(U.std(axis=0).mean())

er = np.array([eff_rank(Z) for Z in embeddings])
sn = np.array([std_norm(Z) for Z in embeddings])

answers = {
    "eff_rank": er,
    "std_norm": sn,
    "n_collapsed": int((er < 5).sum()),
}
```